### Notebook de gráficos

Este notebook toma las tablas generadas por `02_processing.ipynb` en `data/Proc_data/batch_analysis` y construye gráficos del aumento relativo entre alta y baja temperatura por ROI.

Como `low_mean` y `high_mean` son `NormSignal` (`DeltaF/F0`), el ratio principal compara el `F/F0` estimado:

`high_low_ratio = (1 + high_mean) / (1 + low_mean)`

Así, `1` significa que high y low son iguales; valores mayores que `1` indican aumento en alta temperatura respecto al rango bajo.

In [1]:
%matplotlib qt
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path("../scripts").resolve()))
import graphs as gph
gph = importlib.reload(gph)

base_dir = Path("/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data")
batch_dir = base_dir / "batch_analysis"
batch_dir.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

#### Parámetros

Edita esta celda para elegir qué ROIs entran al gráfico.

In [8]:
# Fuente de datos
# "batch_summary" usa roi_temp_summary_active.csv, la salida principal curada por 02_processing.ipynb.
# "preprocessed" usa todos los *_preprocessed_long.csv disponibles, sin la curación manual final.
data_source = "batch_summary"

# Filtros principales
genotype_filter = None      # Ejemplos: None, "m65", ["m27", "m36"]
trend_filter = None         # None muestra todas las categorías en el eje X
roi_status_filter = 1        # 1 = incluidas, 0 = excluidas, None = todas

# Columnas usadas para el ratio
numerator_col = "high_mean"
denominator_col = "low_mean"
ratio_col = "high_low_ratio"
ratio_mode = "relative_total"  # (1 + high_mean) / (1 + low_mean)

# Evita ratios infinitos cuando el denominador está demasiado cerca de cero.
min_abs_denominator = 1e-9

# Category plot
category_col = "trend"
color_col = "sample"
trend_order = ["increase", "stable", "decrease", "insufficient"]
label_roi_points = False  # True para mostrar el nombre de cada ROI sobre los puntos
y_lim = (0.95, 1.5)
figure_size = (3, 5)       # pulgadas: ancho, alto
figure_dpi = 300
save_figures = True
figure_format = "png"

# Exportación
save_outputs = True
output_prefix = "roi_high_low_ratio"

In [11]:
# CSV Data Load
summary_id_cols = [
    "source_folder", "source_file", "sample", "genotype", "genotype_meta",
    "nickname_meta", "ROI", "low_mean", "low_sd", "low_n", "mid_mean", "mid_sd",
    "mid_n", "high_mean", "high_sd", "high_n", "delta_high_low", "trend", "ROI_status",
]

if data_source == "batch_summary":
    summary_path_active = batch_dir / "roi_temp_summary_active.csv"
    summary_path_status = batch_dir / "roi_temp_summary_with_roi_status.csv"

    if summary_path_active.exists():
        roi_summary = pd.read_csv(summary_path_active)
        source_label = str(summary_path_active)
    elif summary_path_status.exists():
        roi_summary = pd.read_csv(summary_path_status)
        source_label = str(summary_path_status)
    else:
        raise FileNotFoundError(
            "No se encontró roi_temp_summary_active.csv ni roi_temp_summary_with_roi_status.csv. "
            "Ejecuta primero 02_processing.ipynb o usa data_source='preprocessed'."
        )
elif data_source == "preprocessed":
    preprocessed_files = sorted(base_dir.glob("*/*_preprocessed_long.csv"))
    if not preprocessed_files:
        raise FileNotFoundError("No se encontraron archivos *_preprocessed_long.csv en data/Proc_data.")

    tables = []
    for file_path in preprocessed_files:
        df = pd.read_csv(file_path)
        df["source_folder"] = file_path.parent.name
        df["source_file"] = file_path.name
        if "genotype_meta" not in df.columns and "genotype" in df.columns:
            df["genotype_meta"] = df["genotype"]
        keep_cols = [col for col in summary_id_cols if col in df.columns]
        tables.append(df[keep_cols].drop_duplicates())

    roi_summary = pd.concat(tables, ignore_index=True).drop_duplicates()
    source_label = f"{len(preprocessed_files)} archivos *_preprocessed_long.csv"
else:
    raise ValueError("data_source debe ser 'preprocessed' o 'batch_summary'.")

print(f"Fuente: {source_label}")
print(f"Filas iniciales: {roi_summary.shape[0]}")
print("Columnas:", list(roi_summary.columns))
if "trend" in roi_summary.columns:
    display(roi_summary["trend"].value_counts(dropna=False).rename("n_roi"))
roi_summary.head()

ratio_df = gph.prepare_ratio_dataframe(
    roi_summary,
    genotype_filter=genotype_filter,
    trend_filter=trend_filter,
    roi_status_filter=roi_status_filter,
    numerator_col=numerator_col,
    denominator_col=denominator_col,
    ratio_col=ratio_col,
    min_abs_denominator=min_abs_denominator,
    ratio_mode=ratio_mode,
)

print("Filtro genotype:", "todos" if genotype_filter is None else genotype_filter)
print("Filtro trend:", "todos" if trend_filter is None else trend_filter)
print("Filtro ROI_status:", "todos" if roi_status_filter is None else roi_status_filter)
print("Ratio:", "(1 + high_mean) / (1 + low_mean)" if ratio_mode == "relative_total" else "high_mean / low_mean")
print(f"ROIs después de filtros: {ratio_df.shape[0]}")
print(f"Ratios válidos: {int(ratio_df['ratio_valid'].sum())}")

if "trend" in ratio_df.columns:
    display(ratio_df["trend"].value_counts(dropna=False).rename("n_roi"))

ratio_df.head()

Fuente: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/roi_temp_summary_active.csv
Filas iniciales: 961
Columnas: ['source_folder', 'source_file', 'sample', 'genotype', 'genotype_meta', 'nickname_meta', 'ROI', 'low_mean', 'low_sd', 'low_n', 'mid_mean', 'mid_sd', 'mid_n', 'high_mean', 'high_sd', 'high_n', 'delta_high_low', 'trend', 'ROI_status']


trend
increase    762
stable      199
Name: n_roi, dtype: int64

Filtro genotype: todos
Filtro trend: todos
Filtro ROI_status: 1
Ratio: (1 + high_mean) / (1 + low_mean)
ROIs después de filtros: 961
Ratios válidos: 961


trend
increase    762
stable      199
Name: n_roi, dtype: int64

,source_folder,source_file,sample,genotype,genotype_meta,nickname_meta,ROI,low_mean,low_sd,low_n,...,high_n,delta_high_low,trend,ROI_status,high_relative_total,low_relative_total,high_low_ratio,raw_normsignal_high_low_ratio,high_minus_low,ratio_valid
0,mut27_image10,sample_10_m27_cooling_preprocessed_long.csv,sample_10,m27,m27,10,ROI1,0.002301,0.006843,38,...,7,0.038968,increase,1,1.041270,1.002301,1.038879,17.931847,0.038968,True
1,mut27_image10,sample_10_m27_cooling_preprocessed_long.csv,sample_10,m27,m27,10,ROI100,0.008420,0.022671,38,...,7,0.022586,increase,1,1.031006,1.008420,1.022398,3.682405,0.022586,True
2,mut27_image10,sample_10_m27_cooling_preprocessed_long.csv,sample_10,m27,m27,10,ROI104,-0.003570,0.009590,38,...,7,0.065612,increase,1,1.062041,0.996430,1.065847,-17.376381,0.065612,True
3,mut27_image10,sample_10_m27_cooling_preprocessed_long.csv,sample_10,m27,m27,10,ROI105,-0.000254,0.007813,38,...,7,0.021537,increase,1,1.021282,0.999746,1.021542,-83.672629,0.021537,True
4,mut27_image10,sample_10_m27_cooling_preprocessed_long.csv,sample_10,m27,m27,10,ROI106,0.011132,0.026295,38,...,7,0.103533,increase,1,1.114665,1.011132,1.102393,10.300145,0.103533,True


#### Gráfico 1: cajas y bigotes por trend

Se genera un gráfico por cada genotipo activo en el filtro. Cada caja resume la distribución de ROIs para una tendencia, y los puntos muestran las ROIs individuales. La línea horizontal en `1` marca respuestas iguales en alta y baja temperatura.

In [12]:
boxplot_axes = gph.plot_ratio_boxplots_by_genotype(
    ratio_df,
    ratio_col=ratio_col,
    trend_col=category_col,
    genotype_col="genotype_meta",
    color_col=color_col,
    trend_order=trend_order,
    label_roi_points=label_roi_points,
    legend_loc="upper right",
    legend_fontsize=5.25,
    legend_title_fontsize=6.3,
    ylim=y_lim,
    figsize=figure_size,
)

if save_figures:
    figure_dir = batch_dir / "figures"
    figure_dir.mkdir(exist_ok=True)
    for genotype, ax in boxplot_axes.items():
        safe_genotype = str(genotype).replace("/", "-").replace(" ", "_")
        fig_path = figure_dir / f"high_low_ratio_boxplot_{safe_genotype}.{figure_format}"
        ax.figure.savefig(fig_path, dpi=figure_dpi, bbox_inches="tight")
        width_px = int(ax.figure.get_figwidth() * figure_dpi)
        height_px = int(ax.figure.get_figheight() * figure_dpi)
        print(f"Guardado: {fig_path} ({ax.figure.get_figwidth():.2f} x {ax.figure.get_figheight():.2f} in, {width_px} x {height_px} px)")

plt.show()

Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/high_low_ratio_boxplot_m27.png (3.00 x 5.00 in, 900 x 1500 px)
Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/high_low_ratio_boxplot_m36.png (3.00 x 5.00 in, 900 x 1500 px)
Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/high_low_ratio_boxplot_m45.png (3.00 x 5.00 in, 900 x 1500 px)
Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/high_low_ratio_boxplot_m57.png (3.00 x 5.00 in, 900 x 1500 px)
Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/high_low_ratio_boxplot_m65.png (3.00 x 5.00 in, 900 x 1500 px)


#### Gráfico 2: low vs high por genotype, sample y trend

Se genera un gráfico por cada genotipo activo. Dentro de cada gráfico, el color identifica `sample` y la forma identifica `trend`. La diagonal marca `high_mean = low_mean`.

In [10]:
low_high_axes = gph.plot_low_vs_high_by_sample_trend(
    ratio_df,
    low_col=denominator_col,
    high_col=numerator_col,
    sample_col="sample",
    trend_col="trend",
    genotype_col="genotype_meta",
    figsize=figure_size,
)

if save_figures:
    figure_dir = batch_dir / "figures"
    figure_dir.mkdir(exist_ok=True)
    for genotype, ax in low_high_axes.items():
        safe_genotype = str(genotype).replace("/", "-").replace(" ", "_")
        fig_path = figure_dir / f"low_vs_high_sample_trend_{safe_genotype}.{figure_format}"
        ax.figure.savefig(fig_path, dpi=figure_dpi, bbox_inches="tight")
        width_px = int(ax.figure.get_figwidth() * figure_dpi)
        height_px = int(ax.figure.get_figheight() * figure_dpi)
        print(f"Guardado: {fig_path} ({ax.figure.get_figwidth():.2f} x {ax.figure.get_figheight():.2f} in, {width_px} x {height_px} px)")

plt.show()

Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/low_vs_high_sample_trend_m27.png (3.00 x 5.00 in, 900 x 1500 px)
Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/low_vs_high_sample_trend_m36.png (3.00 x 5.00 in, 900 x 1500 px)
Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/low_vs_high_sample_trend_m45.png (3.00 x 5.00 in, 900 x 1500 px)
Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/low_vs_high_sample_trend_m57.png (3.00 x 5.00 in, 900 x 1500 px)
Guardado: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/figures/low_vs_high_sample_trend_m65.png (3.00 x 5.00 in, 900 x 1500 px)


#### Resumen por muestra y trend

In [13]:
ratio_summary = gph.summarize_ratio_by_group(
    ratio_df,
    group_cols=("genotype_meta", "sample", "trend"),
    ratio_col=ratio_col,
    numerator_col=numerator_col,
    denominator_col=denominator_col,
)

if ratio_summary.empty:
    print("No hay datos para resumir.")

ratio_summary

,genotype_meta,sample,trend,n_roi,ratio_mean,ratio_sd,ratio_sem,high_mean,low_mean,delta_mean
0,m27,sample_10,increase,185,1.044822,0.045066,0.003313,0.045791,0.000909,0.044882
1,m27,sample_10,stable,115,1.000365,0.005567,0.000519,0.003850,0.003483,0.000367
2,m27,sample_11,increase,182,1.073916,0.057527,0.004264,0.075066,0.001151,0.073915
3,m27,sample_11,stable,53,1.001528,0.005596,0.000769,0.007040,0.005502,0.001538
4,m36,sample_08,increase,276,1.086451,0.023961,0.001442,0.075857,-0.009767,0.085624
5,m36,sample_08,stable,13,1.002619,0.005138,0.001425,-0.011742,-0.014298,0.002556
6,m45,sample_13,increase,70,1.202299,0.077353,0.009245,0.197826,-0.003891,0.201717
7,m45,sample_13,stable,4,0.998369,0.002778,0.001389,-0.009544,-0.007935,-0.001609
8,m57,sample_02,increase,6,1.043694,0.011741,0.004793,0.042673,-0.000983,0.043656
9,m57,sample_02,stable,7,1.000392,0.005217,0.001972,-0.001383,-0.001773,0.000390


In [ ]:
if save_outputs:
    suffix_parts = [data_source]
    if genotype_filter is not None:
        suffix_parts.append("genotype-" + "_".join(genotype_filter if isinstance(genotype_filter, list) else [str(genotype_filter)]))
    if trend_filter is not None:
        suffix_parts.append(f"trend-{trend_filter}")
    if roi_status_filter is not None:
        suffix_parts.append(f"roi_status-{roi_status_filter}")
    suffix = "_" + "_".join(suffix_parts) if suffix_parts else "_all"

    ratio_path = batch_dir / f"{output_prefix}{suffix}.csv"
    summary_path = batch_dir / f"{output_prefix}_summary{suffix}.csv"

    ratio_df.to_csv(ratio_path, index=False)
    ratio_summary.to_csv(summary_path, index=False)

    print("Exportado:")
    print(f"  ratios: {ratio_path}")
    print(f"  resumen: {summary_path}")